<a href="https://colab.research.google.com/github/nagmanisha967-ops/my-projects/blob/main/ManishaNag_logistic_regression__project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



*   **Project Name:** Diabetes Prediction using Kaggle Data

*   **Student Name-** Manisha Nag


*   **Date:** 04/10/2026
*   Time - 1:00pm






In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# 1. Load Kaggle CSV dataset
df = pd.read_csv('diabetes_prediction_dataset.csv.zip')

# 2. Inspect data
print("Dataset Shape:", df.shape)
display(df.head())
print(df.info())

# 3. Handle zero-replacement for valid clinical features if appropriate.
# In this dataset, HbA1c and Blood Glucose levels are not zero for valid rows.
# If you wish to replace zeros in 'bmi' with NaN, you can do so here:
df['bmi'] = df['bmi'].replace(0, np.nan)

# Define Features (X) and Target (y)
# The target label in this dataset is 'diabetes'
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# Encode categorical variables ('gender' and 'smoking_history')
X = pd.get_dummies(X, columns=['gender', 'smoking_history'], drop_first=True)

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Median Imputation (fit on training data to prevent leakage)
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 6. Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# 7. Train Balanced Logistic Regression
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# 8. Evaluate
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\n--- Performance Metrics ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall: {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# 9. Interpret Coefficients
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print("\n--- Model Coefficients ---")
print(coefficients)